# Hyperion public market campaign — evidence review

Read-only analysis of run `a43cc042-9fee-424b-87ba-b44457b16b7c`, collected on 2026-09-20.
The 2026-09-21 server inspection is recorded in `acceptance.json`. No new collection is started.

Copy the 32 public JSON files from the documented Hyperion run directory into a local directory; set `CRYPTO_OBSERVATION_RUN_DIR` if different from the default. Run from this checkout or a subdirectory. Requires existing Node dependencies and Python standard library only. Raw books stay outside Git.
Cell assertions intentionally apply to this accepted run; they are not generic statistical guarantees. Timestamp units are epoch milliseconds; quantiles use nearest rank.


In [1]:
import hashlib
import json
import math
import os
import statistics
import subprocess
from pathlib import Path

# Only the observation path is configurable. No credentials or private API calls.
DATA = Path(os.environ.get('CRYPTO_OBSERVATION_RUN_DIR', '/tmp/crypto-campaign-20260920-review'))
REPO = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p/'src/scripts/market-campaign.ts').is_file())
EVIDENCE = REPO/'docs/evidence/market-campaign-20260920'
acceptance = json.loads((EVIDENCE/'acceptance.json').read_text())
hashes = {name: hashlib.sha256((DATA/name).read_bytes()).hexdigest() for name in acceptance['files']}
assert hashes == acceptance['files'], 'Evidence file hashes differ from the verified Hyperion copy'
manifest = json.loads((DATA/'run.json').read_text())
samples = [json.loads((DATA/f'{i:03}.json').read_text()) for i in range(manifest['samples'])]
print(f"Verified {len(hashes)} files for {manifest['runId']}")


Verified 32 files for a43cc042-9fee-424b-87ba-b44457b16b7c


In [2]:
# Existing TypeScript validators perform the canonical schema, book, size,
# source freshness and comparison checks. This subprocess has no network work.
process = subprocess.run([str(REPO/'node_modules/.bin/tsx'),
    'src/scripts/market-campaign.ts', 'report', str(DATA.resolve())],
    cwd=REPO, check=True, capture_output=True, text=True)
report = json.loads(process.stdout)
assert report == acceptance['api']['report'], 'Offline calculation no longer matches deployed report'
keys = [(s['runId'], s['sequence'], source['venue']) for s in samples for source in s['sources']]
assert len(keys) == len(set(keys)) == 90
assert [s['sequence'] for s in samples] == list(range(30))
assert report['collection']['state'] == 'completed'
assert not report['missingSequences']
assert report['recordedSamples'] == report['expectedSamples'] == 30
assert all(v['received'] == v['freshAtComparison'] == 30 and v['failed'] == 0 for v in report['byVenue'].values())
assert all(p['valid'] == p['sizeChecked'] == 30 and p['rejected'] == 0 for p in report['pairs'].values())
print('Canonical validation, unique grain, coverage and instrument-size checks passed.')


Canonical validation, unique grain, coverage and instrument-size checks passed.


In [3]:
def describe(values):
    values = sorted(values)
    if not values:
        return None
    return {'count': len(values), 'min': values[0], 'median': statistics.median(values),
            'p95NearestRank': values[math.ceil(.95*len(values))-1], 'max': values[-1]}

timing = {}
for venue in ['binance', 'bybit', 'okx']:
    books = [source['book'] for s in samples for source in s['sources']
             if source['venue'] == venue and source['available']]
    timing[venue] = {
        'requestDurationMs': describe([b['receivedAt']-b['requestedAt'] for b in books]),
        'sourceToReceiptMs': describe([b['receivedAt']-b['sourceAt'] for b in books if 'sourceAt' in b]),
        'sourceTimePresent': sum('sourceAt' in b for b in books),
    }
windows = {pair: [] for pair in report['pairs']}
for s in samples:
    books = {source['venue']: source['book'] for source in s['sources'] if source['available']}
    for pair in windows:
        buy, sell = (books[v] for v in pair.split('->'))
        # Same conservative observed window as compareVenues, not simultaneity proof.
        earliest = min(buy.get('sourceAt', buy['requestedAt']), sell.get('sourceAt', sell['requestedAt']))
        latest = max(buy['receivedAt'], sell['receivedAt'])
        windows[pair].append(latest-earliest)
assert all(max(v) <= 2000 for v in windows.values())
quality = {
    'schema': 1, 'runId': report['runId'], 'sourceHost': acceptance['sourceHost'],
    'sourceRun': acceptance['sourceRun'], 'deployedCommit': acceptance['deployedCommit'],
    'grain': 'one run/sequence/venue book; each sequence yields six directional comparisons',
    'window': {'startedAt': report['startedAt'], 'lastObservedAt': report['lastObservedAt'],
               'elapsedMs': report['lastObservedAt']-report['startedAt']},
    'checks': {'copiedSourceHashesMatch': hashes == acceptance['files'],
               'apiMatchesOfflineReport': report == acceptance['api']['report'],
               'uniqueBookKeys': len(set(keys)), 'duplicateBookKeys': len(keys)-len(set(keys)),
               'samples': report['recordedSamples'], 'missingSequences': report['missingSequences'],
               'freshBooks': sum(v['freshAtComparison'] for v in report['byVenue'].values()),
               'sourceFailures': sum(v['failed'] for v in report['byVenue'].values()),
               'validDirectionalComparisons': sum(p['valid'] for p in report['pairs'].values()),
               'sizeCheckedComparisons': sum(p['sizeChecked'] for p in report['pairs'].values()),
               'rejectedComparisons': sum(p['rejected'] for p in report['pairs'].values())},
    'sampleStartGapsMs': describe([b['startedAt']-a['startedAt'] for a,b in zip(samples,samples[1:])]),
    'batchDurationMs': describe([s['checkedAt']-s['startedAt'] for s in samples]),
    'venueTiming': timing, 'pairWindowMs': {pair: describe(v) for pair,v in windows.items()},
    'costAssumptions': report['assumptions'],
    'result': {'positiveDirectionalComparisons': sum(p['positive'] for p in report['pairs'].values()),
               'bestNetBps': max(p['bestNetBps'] for p in report['pairs'].values()),
               'worstNetBps': min(p['worstNetBps'] for p in report['pairs'].values())},
    'decision': 'pass for functional offline replay; insufficient for profitability or live execution claims',
    'limitations': ['30 snapshots over roughly 29 minutes; no multi-day or multi-regime coverage',
                    'Binance has no source timestamp; receipt timing does not prove matching-engine freshness',
                    'six directional comparisons share snapshots and are not independent trades',
                    'fees/slippage are illustrative; no account eligibility, tariffs, funding or rebalancing verified',
                    'source-to-receipt timing includes unverified cross-host clock differences',
                    'fixed quantity; public instrument rules are estimates, not exchange order acceptance']
}
print(json.dumps(quality, indent=2))


{
  "schema": 1,
  "runId": "a43cc042-9fee-424b-87ba-b44457b16b7c",
  "sourceHost": "Hyperion",
  "sourceRun": "/home/mil/crypto-market-observations/store/runs/a43cc042-9fee-424b-87ba-b44457b16b7c",
  "deployedCommit": "386ee2c1734af87f76ab3745b1076739cd5195f4",
  "grain": "one run/sequence/venue book; each sequence yields six directional comparisons",
  "window": {
    "startedAt": 1789887959214,
    "lastObservedAt": 1789889700152,
    "elapsedMs": 1740938
  },
  "checks": {
    "copiedSourceHashesMatch": true,
    "apiMatchesOfflineReport": true,
    "uniqueBookKeys": 90,
    "duplicateBookKeys": 0,
    "samples": 30,
    "missingSequences": [],
    "freshBooks": 90,
    "sourceFailures": 0,
    "validDirectionalComparisons": 180,
    "sizeCheckedComparisons": 180,
    "rejectedComparisons": 0
  },
  "sampleStartGapsMs": {
    "count": 29,
    "min": 58326,
    "median": 60000,
    "p95NearestRank": 60035,
    "max": 60086
  },
  "batchDurationMs": {
    "count": 30,
    "min": 349,

## Interpretation

Full coverage and deterministic reconstruction support functional paper-v2 replay development. They do not establish sustainable profitability. No positive spread survived the stated costs. Latency, source timestamps and instrument rules qualify these comparisons; they cannot prove two executable legs. The existing paper journal and all trading controls were unchanged.
